# Neural Axis BCI Quick Start Guide

Welcome to the Neural Axis Brain-Computer Interface emotion recognition system! This notebook will guide you through the complete workflow from data preparation to real-time emotion prediction.

## What You'll Learn
1. How to set up different EEG devices
2. Data preprocessing and feature extraction
3. Model training with IIT Φ integration
4. ONNX model export for deployment
5. Real-time emotion prediction

## Prerequisites
- Python 3.8+
- EEG data in EEGLAB .set format (or sample data)
- Basic understanding of EEG and emotion recognition

## Step 1: Environment Setup

First, let's install the required dependencies and import necessary libraries.

In [ ]:
# Install core dependencies (uncomment if needed)
# !pip install -r requirements.txt

# For IIT Φ calculation (optional)
# !pip install -r requirements_phi.txt

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.append(str(project_root))

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

# Verify we can import the required modules
try:
    from scripts.device_adapter import DeviceAdapter
    from scripts.enhanced_features import EnhancedFeatureExtractor
    from scripts.phi_estimator import PhiEstimator
    print("✅ All required modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure you're running from the project root directory")

## Step 2: Device Configuration

The system supports multiple EEG devices. Let's explore the available configurations and set up your device.

In [ ]:
# Import device adapter
from scripts.device_adapter import DeviceAdapter

# Initialize device adapter
device_adapter = DeviceAdapter()

# Available devices (from the actual device configurations)
devices = ["Muse2", "X.on", "OpenBCI_Cyton", "Standard_10_20"]

print("Available EEG Devices:")
print("=" * 50)

for device_name in devices:
    try:
        # Set device and get info
        if device_adapter.set_device(device_name):
            info = device_adapter.get_device_info(device_name)
            
            print(f"\n📱 {device_name}")
            print(f"   Channels: {len(info.get('channels', []))} ({', '.join(info.get('channels', [])[:4])}{'...' if len(info.get('channels', [])) > 4 else ''})")
            print(f"   Sampling Rate: {info.get('sampling_rate', 'N/A')} Hz")
            faa_config = info.get('faa_channels', {})
            if isinstance(faa_config, dict):
                faa_str = f"{faa_config.get('left', 'N/A')}, {faa_config.get('right', 'N/A')}"
            else:
                faa_str = str(faa_config)
            print(f"   FAA Channels: {faa_str}")
            print(f"   Description: {info.get('description', 'No description available')}")
        else:
            print(f"\n❌ {device_name}: Device not supported")
        
    except Exception as e:
        print(f"❌ Error loading {device_name}: {e}")

# Set your device (change this to match your hardware)
DEVICE_NAME = "Muse2"  # Change to your device

if device_adapter.set_device(DEVICE_NAME):
    print(f"\n✅ Using device: {DEVICE_NAME}")
    info = device_adapter.get_device_info(DEVICE_NAME)
    print(f"Channels: {info.get('channels', [])}")
    
    # Get FAA channel indices
    faa_left, faa_right = device_adapter.get_faa_channel_indices(DEVICE_NAME)
    if faa_left is not None and faa_right is not None:
        channels = info.get('channels', [])
        faa_channels = [channels[faa_left], channels[faa_right]] if faa_left < len(channels) and faa_right < len(channels) else []
        print(f"FAA channels: {faa_channels}")
    else:
        print("FAA channels: Not available")
else:
    print(f"❌ Failed to set device: {DEVICE_NAME}")
    DEVICE_NAME = "Standard_10_20"  # Fallback
    device_adapter.set_device(DEVICE_NAME)
    print(f"Using fallback device: {DEVICE_NAME}")

## Step 3: Data Preparation

Let's prepare some sample data or load your own EEG recordings.

In [ ]:
# Generate sample EEG data (replace with your actual data loading)
def generate_sample_eeg_data(n_channels=4, duration=10, sampling_rate=256):
    """Generate realistic EEG-like data for demonstration"""
    n_samples = int(duration * sampling_rate)
    t = np.linspace(0, duration, n_samples)
    
    data = np.zeros((n_channels, n_samples))
    
    for ch in range(n_channels):
        # Base EEG-like signal with multiple frequency components
        signal = (
            0.5 * np.sin(2 * np.pi * 10 * t + ch * np.pi/4) +  # Alpha
            0.3 * np.sin(2 * np.pi * 20 * t + ch * np.pi/6) +  # Beta
            0.2 * np.sin(2 * np.pi * 6 * t + ch * np.pi/8) +   # Theta
            0.1 * np.random.randn(n_samples)  # Noise
        )
        
        # Add some channel-specific variations
        if ch < 2:  # Left hemisphere bias for positive emotion
            signal += 0.2 * np.sin(2 * np.pi * 10 * t)
        
        data[ch] = signal
    
    return data, t

# Generate sample data
info = device_adapter.get_device_info(DEVICE_NAME)
channels = info.get('channels', ['C3', 'C4', 'Cz', 'Pz'])
sampling_rate = info.get('sampling_rate', 256)

sample_data, time_axis = generate_sample_eeg_data(
    n_channels=len(channels), 
    duration=5, 
    sampling_rate=sampling_rate
)

print(f"Generated sample EEG data:")
print(f"Shape: {sample_data.shape} (channels × samples)")
print(f"Duration: {sample_data.shape[1] / sampling_rate:.1f} seconds")
print(f"Channels: {channels}")

# Visualize the data
plt.figure(figsize=(12, 6))
for i, ch_name in enumerate(channels):
    plt.subplot(len(channels), 1, i+1)
    plt.plot(time_axis, sample_data[i], linewidth=0.8)
    plt.ylabel(f'{ch_name}\n(μV)', rotation=0, labelpad=20)
    plt.grid(True, alpha=0.3)
    if i == 0:
        plt.title('Sample EEG Data')
    if i == len(channels) - 1:
        plt.xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

## Step 4: Feature Extraction

Extract sophisticated features including frequency band powers, differential entropy, and frontal alpha asymmetry.

In [ ]:
# Import enhanced feature extractor
from scripts.enhanced_features import EnhancedFeatureExtractor

# Initialize feature extractor
feature_extractor = EnhancedFeatureExtractor(
    fs=sampling_rate,
    config_path="configs/feature_extraction.yaml"
)

# Find FAA channel indices
faa_left, faa_right = device_adapter.get_faa_channel_indices(DEVICE_NAME)
if faa_left is not None and faa_right is not None:
    faa_indices = (faa_left, faa_right)
    print(f"FAA channels: {channels[faa_left]} (idx {faa_left}) vs {channels[faa_right]} (idx {faa_right})")
else:
    faa_indices = None
    print("No FAA channels found for this device")

# Extract features from a window of data
window_samples = int(2.0 * sampling_rate)
data_window = sample_data[:, :window_samples]

# Use the correct method for feature extraction
spec_features, de_features = feature_extractor.extract_features(
    data_window, 
    faa_channels=faa_indices
)

print("\nExtracted Features:")
print("=" * 30)

# Display spectrogram features shape
print(f"\nSpectrogram features shape: {spec_features.shape}")
print(f"Differential entropy features shape: {de_features.shape}")

# Display sample DE features
print(f"\nSample DE features (first 10): {de_features[:10]}")

# For demonstration, let's also try the structured features method if available
try:
    # Check if the method exists and try to use it
    if hasattr(feature_extractor, 'extract_features_structured'):
        features_structured = feature_extractor.extract_features_structured(
            data_window, 
            faa_channels=faa_indices
        )
        
        print("\nStructured Features:")
        print("=" * 30)
        
        # Display band powers
        if 'band_powers' in features_structured:
            print("\nFrequency Band Powers:")
            for band, powers in features_structured['band_powers'].items():
                avg_power = np.mean(powers)
                print(f"  {band:8s}: {avg_power:8.4f} (avg across channels)")

        # Display differential entropy
        if 'differential_entropy' in features_structured:
            print("\nDifferential Entropy:")
            for band, de_values in features_structured['differential_entropy'].items():
                avg_de = np.mean(de_values)
                print(f"  {band:8s}: {avg_de:8.4f} (avg across channels)")

        # Display FAA
        if 'faa' in features_structured:
            faa_value = features_structured['faa']
            emotion_bias = "Positive" if faa_value > 0 else "Negative" if faa_value < 0 else "Neutral"
            print(f"\nFrontal Alpha Asymmetry: {faa_value:.4f} ({emotion_bias} bias)")

        # Display spectrograms shape
        if 'spectrograms' in features_structured:
            spec_shape = features_structured['spectrograms'].shape
            print(f"\nSpectrograms shape: {spec_shape} (channels × freq × time)")
    else:
        print("Structured features method not available, using basic extraction")
        
except Exception as e:
    print(f"Note: Structured features extraction failed: {e}")
    print("Using basic feature extraction results above")

## Step 5: IIT Φ (Phi) Calculation

Demonstrate the Integrated Information Theory (IIT) consciousness measure calculation.

In [ ]:
# Import IIT Φ estimator
try:
    from scripts.phi_estimator import PhiEstimator
    import torch
    TORCH_AVAILABLE = True
    print("✅ PyTorch and PhiEstimator available")
except ImportError as e:
    print(f"⚠️ PyTorch not available: {e}")
    print("   PhiEstimator will use mock mode only")
    TORCH_AVAILABLE = False

# Initialize Φ estimator
if TORCH_AVAILABLE:
    phi_estimator = PhiEstimator()
else:
    # Create a mock estimator for demonstration
    class MockPhiEstimator:
        def compute(self, data):
            import numpy as np
            # Return a mock phi value
            return np.random.rand() * 0.1 + 0.01
    
    phi_estimator = MockPhiEstimator()
    print("Using mock PhiEstimator (install PyTorch for real Φ calculation)")

print("IIT Φ (Integrated Information) Calculation")
print("=" * 45)

# Test different Φ calculation methods
methods = ['mock', 'IIT3.0', 'IIT4.0_light']

for method in methods:
    print(f"\n🧠 Testing {method} method:")
    
    try:
        if TORCH_AVAILABLE:
            # Convert data to torch tensor (PhiEstimator expects torch tensors)
            data_tensor = torch.tensor(data_window, dtype=torch.float32).unsqueeze(0)  # Add batch dimension
            
            # Calculate Φ for the data window
            phi_value = phi_estimator.compute(data_tensor)
            
            # Extract scalar value
            phi_scalar = phi_value.item() if hasattr(phi_value, 'item') else float(phi_value)
        else:
            # Use mock calculation
            phi_scalar = phi_estimator.compute(data_window)
        
        print(f"   Φ = {phi_scalar:.6f}")
        
        # Interpret the value
        if phi_scalar > 0.1:
            interpretation = "High consciousness/integration"
        elif phi_scalar > 0.05:
            interpretation = "Moderate consciousness/integration"
        elif phi_scalar > 0.01:
            interpretation = "Low consciousness/integration"
        else:
            interpretation = "Minimal/no consciousness"
        
        print(f"   Interpretation: {interpretation}")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Demonstrate Φ over time
print("\n📈 Φ values over time windows:")
window_size = int(1.0 * sampling_rate)  # 1-second windows
hop_size = window_size // 2  # 50% overlap

phi_values = []
timestamps = []

for start_idx in range(0, sample_data.shape[1] - window_size, hop_size):
    end_idx = start_idx + window_size
    window = sample_data[:, start_idx:end_idx]
    
    try:
        if TORCH_AVAILABLE:
            # Convert to torch tensor
            window_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0)
            phi_val = phi_estimator.compute(window_tensor)
            phi_scalar = phi_val.item() if hasattr(phi_val, 'item') else float(phi_val)
        else:
            # Use mock calculation
            phi_scalar = phi_estimator.compute(window)
        
        phi_values.append(phi_scalar)
        timestamps.append(start_idx / sampling_rate)
    except Exception as e:
        print(f"Error computing Φ at {start_idx/sampling_rate:.1f}s: {e}")
        phi_values.append(0.0)
        timestamps.append(start_idx / sampling_rate)

# Plot Φ over time
if phi_values:
    plt.figure(figsize=(12, 4))
    plt.plot(timestamps, phi_values, 'b-o', linewidth=2, markersize=4)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Φ (Integrated Information)')
    plt.title('IIT Φ Values Over Time')
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f"Average Φ: {np.mean(phi_values):.6f} ± {np.std(phi_values):.6f}")
else:
    print("No Φ values computed successfully")

## Step 6: Model Training Simulation

Demonstrate the training process with sample data and labels.

In [ ]:
# Create sample training data
def create_sample_dataset(n_samples=50, device_adapter=device_adapter, channels=channels, sampling_rate=sampling_rate):
    """Create a sample dataset for demonstration"""
    samples = []
    labels = []
    phi_values = []
    
    for i in range(n_samples):
        # Generate varied EEG-like data
        emotion_bias = np.random.uniform(-1, 1)  # Random emotion state
        arousal_level = np.random.uniform(-1, 1)  # Random arousal level
        
        data, _ = generate_sample_eeg_data(
            n_channels=len(channels),
            duration=2,
            sampling_rate=sampling_rate
        )
        
        # Add emotion-dependent variations
        if emotion_bias > 0:  # Positive emotion - more left alpha
            data[0] *= (1 + 0.3 * emotion_bias)  # Boost left channel
        else:  # Negative emotion - more right alpha
            data[1] *= (1 + 0.3 * abs(emotion_bias))  # Boost right channel
        
        samples.append(data)
        labels.append([emotion_bias, arousal_level])  # [valence, arousal]
        
        # Simulate Φ calculation
        try:
            if TORCH_AVAILABLE:
                data_tensor = torch.tensor(data, dtype=torch.float32).unsqueeze(0)
                phi_val = phi_estimator.compute(data_tensor)
                phi_scalar = phi_val.item() if hasattr(phi_val, 'item') else float(phi_val)
            else:
                phi_scalar = phi_estimator.compute(data)
            
            phi_values.append(phi_scalar)
        except Exception as e:
            print(f"Error computing Φ for sample {i}: {e}")
            phi_values.append(0.0)
    
    return samples, np.array(labels), np.array(phi_values)

# Create sample dataset
print("Creating sample training dataset...")
train_samples, train_labels, train_phi = create_sample_dataset(n_samples=20)

print(f"Created {len(train_samples)} training samples")
print(f"Label range - Valence: [{train_labels[:, 0].min():.2f}, {train_labels[:, 0].max():.2f}]")
print(f"Label range - Arousal: [{train_labels[:, 1].min():.2f}, {train_labels[:, 1].max():.2f}]")
print(f"Φ range: [{train_phi.min():.6f}, {train_phi.max():.6f}]")

# Visualize the emotion distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(train_labels[:, 0], train_labels[:, 1], c=train_phi, cmap='viridis', s=50, alpha=0.7)
plt.colorbar(label='Φ (Integrated Information)')
plt.xlabel('Valence')
plt.ylabel('Arousal')
plt.title('Emotion Distribution with Φ Values')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(train_phi, bins=15, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Φ (Integrated Information)')
plt.ylabel('Frequency')
plt.title('Distribution of Φ Values')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Sample dataset created successfully!")
print("📝 Note: In real usage, you would:")
print("   1. Load actual EEG .set files from your data directory")
print("   2. Run: python train/train_labeled.py --data_dir your_data --epochs 50")
print("   3. Enable Φ computation with --compute_phi flag")
print("   4. Use cross-validation with --cv_method LOSO")

## Step 7: Model Export and Deployment

Demonstrate how to export trained models to ONNX format for deployment.

In [ ]:
# Check if a pre-trained model exists
model_path = project_root / "model" / "model_onnx" / "va_regressor.onnx"

if model_path.exists():
    print(f"✅ Found existing ONNX model: {model_path}")
    
    # Test ONNX model loading
    try:
        import onnxruntime as ort
        
        session = ort.InferenceSession(str(model_path))
        
        print("\nModel Information:")
        print(f"  Inputs: {len(session.get_inputs())}")
        for i, input_detail in enumerate(session.get_inputs()):
            print(f"    {i+1}. {input_detail.name}: {input_detail.shape}")
        
        print(f"  Outputs: {len(session.get_outputs())}")
        for i, output_detail in enumerate(session.get_outputs()):
            print(f"    {i+1}. {output_detail.name}: {output_detail.shape}")
        
        # Test inference with sample data
        print("\n🧪 Testing model inference...")
        
        # Prepare inputs based on the actual model architecture
        # The model expects spectrogram and differential entropy features
        try:
            # Use the features we extracted earlier
            if 'spec_features' in locals() and 'de_features' in locals():
                # Prepare spectrogram input (assuming 3 channels, 224x224)
                if len(spec_features.shape) == 3:  # (channels, height, width)
                    spec_input = spec_features[np.newaxis, :, :, :].astype(np.float32)
                else:
                    # Create dummy spectrogram if shape doesn't match
                    spec_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
                
                # Prepare DE input
                de_input = de_features[np.newaxis, :].astype(np.float32)
                
                # Pad or truncate DE features to match expected input size
                expected_de_size = 26  # Adjust based on your model
                if de_input.shape[1] < expected_de_size:
                    padding = np.zeros((1, expected_de_size - de_input.shape[1]))
                    de_input = np.concatenate([de_input, padding], axis=1)
                elif de_input.shape[1] > expected_de_size:
                    de_input = de_input[:, :expected_de_size]
                
                # Prepare inputs dictionary
                inputs = {}
                for i, input_detail in enumerate(session.get_inputs()):
                    if 'spec' in input_detail.name.lower() or i == 0:
                        inputs[input_detail.name] = spec_input
                    else:
                        inputs[input_detail.name] = de_input
                
                outputs = session.run(None, inputs)
                prediction = outputs[0][0]  # Get first prediction
                
                valence, arousal = prediction[0], prediction[1]
                
                print(f"   Predicted Valence: {valence:.3f} ({'Positive' if valence > 0 else 'Negative'})")
                print(f"   Predicted Arousal: {arousal:.3f} ({'High' if arousal > 0 else 'Low'})")
                
                # Emotion quadrant mapping
                if valence > 0 and arousal > 0:
                    emotion = "Happy/Excited"
                elif valence > 0 and arousal < 0:
                    emotion = "Calm/Relaxed"
                elif valence < 0 and arousal > 0:
                    emotion = "Angry/Stressed"
                else:
                    emotion = "Sad/Depressed"
                
                print(f"   Emotion State: {emotion}")
                
                print("\n✅ Model inference successful!")
            else:
                print("❌ No features available for testing. Run feature extraction first.")
                
        except Exception as e:
            print(f"❌ Feature preparation failed: {e}")
            print("Using dummy data for demonstration...")
            
            # Fallback to dummy data
            dummy_spec = np.random.randn(1, 3, 224, 224).astype(np.float32)
            dummy_de = np.random.randn(1, 26).astype(np.float32)
            
            inputs = {
                session.get_inputs()[0].name: dummy_spec,
                session.get_inputs()[1].name: dummy_de
            }
            
            outputs = session.run(None, inputs)
            prediction = outputs[0][0]
            
            valence, arousal = prediction[0], prediction[1]
            print(f"   Dummy Prediction - Valence: {valence:.3f}, Arousal: {arousal:.3f}")
        
    except ImportError:
        print("❌ onnxruntime not installed. Install with: pip install onnxruntime")
    except Exception as e:
        print(f"❌ Model test failed: {e}")

else:
    print(f"❌ No ONNX model found at {model_path}")
    print("\n📝 To create a model:")
    print("   1. Train a model: python scripts/train/train_labeled.py --data_dir your_data")
    print("   2. Export to ONNX: python scripts/tools/export_onnx.py --weights model.pth")

print("\n📋 ONNX Export Command Examples:")
print(f"   python scripts/tools/export_onnx.py --weights model/model_weight/ckpt.pt")
print(f"   python scripts/tools/export_onnx.py --weights model.pth --model_type EfficientNet")
print(f"   python scripts/tools/inference_benchmark.py --model {model_path} --provider cuda")

## Step 8: Real-time Emotion Recognition Simulation

Simulate real-time emotion recognition with live data processing.

In [ ]:
import time
from collections import deque
import threading

class EmotionRecognitionSimulator:
    """Simulates real-time emotion recognition"""
    
    def __init__(self, device_adapter, feature_extractor, phi_estimator, channels, sampling_rate):
        self.device_adapter = device_adapter
        self.feature_extractor = feature_extractor
        self.phi_estimator = phi_estimator
        self.channels = channels
        self.sampling_rate = sampling_rate
        
        self.buffer_size = int(2 * sampling_rate)  # 2-second buffer
        self.data_buffer = deque(maxlen=self.buffer_size)
        
        self.results = []
        self.running = False
    
    def add_data_chunk(self, chunk):
        """Add new data chunk to buffer"""
        for sample in chunk.T:  # Add sample by sample
            self.data_buffer.append(sample)
    
    def process_buffer(self):
        """Process current buffer content"""
        if len(self.data_buffer) < self.buffer_size:
            return None
        
        # Convert buffer to array
        data_array = np.array(list(self.data_buffer)).T  # Transpose to (channels, samples)
        
        # Extract features
        faa_left, faa_right = self.device_adapter.get_faa_channel_indices(DEVICE_NAME)
        if faa_left is not None and faa_right is not None:
            faa_indices = (faa_left, faa_right)
        else:
            faa_indices = None
        
        try:
            # Use the correct feature extraction method
            spec_features, de_features = self.feature_extractor.extract_features(
                data_array, faa_channels=faa_indices
            )
            
            # Calculate Φ
            if TORCH_AVAILABLE:
                data_tensor = torch.tensor(data_array, dtype=torch.float32).unsqueeze(0)
                phi_value = self.phi_estimator.compute(data_tensor)
                phi_scalar = phi_value.item() if hasattr(phi_value, 'item') else float(phi_value)
            else:
                phi_scalar = self.phi_estimator.compute(data_array)
            
            # Simple heuristic prediction based on features
            # Use DE features for emotion estimation
            de_mean = np.mean(de_features)
            de_std = np.std(de_features)
            
            # Simple heuristic prediction
            valence = np.tanh((de_mean - 0.5) * 2)  # DE mean -> valence
            arousal = np.tanh(de_std * 3)  # DE std -> arousal
            
            result = {
                'timestamp': time.time(),
                'valence': float(valence),
                'arousal': float(arousal),
                'phi': float(phi_scalar),
                'de_mean': float(de_mean),
                'de_std': float(de_std)
            }
            
            return result
            
        except Exception as e:
            print(f"Error processing buffer: {e}")
            return None
    
    def simulate_realtime(self, duration=10, update_rate=4):
        """Simulate real-time processing"""
        print(f"🚀 Starting real-time simulation for {duration} seconds at {update_rate} Hz")
        
        self.results = []
        self.running = True
        
        chunk_size = int(self.sampling_rate / update_rate)
        n_channels = len(self.channels)
        
        start_time = time.time()
        
        while self.running and (time.time() - start_time) < duration:
            # Generate new data chunk
            chunk, _ = generate_sample_eeg_data(
                n_channels=n_channels,
                duration=chunk_size / self.sampling_rate,
                sampling_rate=self.sampling_rate
            )
            
            # Add to buffer
            self.add_data_chunk(chunk)
            
            # Process if buffer is full
            result = self.process_buffer()
            if result:
                self.results.append(result)
                
                # Print current state
                elapsed = result['timestamp'] - start_time
                emotion_state = self.get_emotion_label(result['valence'], result['arousal'])
                
                print(f"[{elapsed:5.1f}s] {emotion_state:12s} | "
                      f"V={result['valence']:+.3f} A={result['arousal']:+.3f} | "
                      f"Φ={result['phi']:.6f} | DE={result['de_mean']:+.3f}")
            
            # Wait for next update
            time.sleep(1.0 / update_rate)
        
        self.running = False
        print(f"\n✅ Simulation completed! Processed {len(self.results)} updates.")
    
    def get_emotion_label(self, valence, arousal):
        """Map valence/arousal to emotion label"""
        if valence > 0.2 and arousal > 0.2:
            return "Happy"
        elif valence > 0.2 and arousal < -0.2:
            return "Calm"
        elif valence < -0.2 and arousal > 0.2:
            return "Angry"
        elif valence < -0.2 and arousal < -0.2:
            return "Sad"
        else:
            return "Neutral"

# Run simulation
simulator = EmotionRecognitionSimulator(device_adapter, feature_extractor, phi_estimator, channels, sampling_rate)
simulator.simulate_realtime(duration=8, update_rate=2)  # 8 seconds at 2 Hz

# Visualize results
if simulator.results:
    results_df = {key: [r[key] for r in simulator.results] for key in simulator.results[0].keys()}
    timestamps = [(t - simulator.results[0]['timestamp']) for t in results_df['timestamp']]
    
    plt.figure(figsize=(14, 8))
    
    # Valence and Arousal
    plt.subplot(2, 2, 1)
    plt.plot(timestamps, results_df['valence'], 'b-o', label='Valence', linewidth=2)
    plt.plot(timestamps, results_df['arousal'], 'r-s', label='Arousal', linewidth=2)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Emotion Value')
    plt.title('Real-time Emotion Recognition')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Phi values
    plt.subplot(2, 2, 2)
    plt.plot(timestamps, results_df['phi'], 'g-^', linewidth=2, markersize=6)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Φ (Integrated Information)')
    plt.title('Consciousness Measure Over Time')
    plt.grid(True, alpha=0.3)
    
    # Valence-Arousal plane
    plt.subplot(2, 2, 3)
    colors = plt.cm.viridis(np.linspace(0, 1, len(results_df['valence'])))
    scatter = plt.scatter(results_df['valence'], results_df['arousal'], 
                         c=results_df['phi'], cmap='viridis', s=60, alpha=0.8)
    plt.colorbar(scatter, label='Φ')
    plt.xlabel('Valence')
    plt.ylabel('Arousal')
    plt.title('Emotion Space Trajectory')
    plt.grid(True, alpha=0.3)
    
    # Add quadrant labels
    plt.text(0.7, 0.7, 'Happy', fontsize=10, alpha=0.7)
    plt.text(0.7, -0.7, 'Calm', fontsize=10, alpha=0.7)
    plt.text(-0.7, 0.7, 'Angry', fontsize=10, alpha=0.7)
    plt.text(-0.7, -0.7, 'Sad', fontsize=10, alpha=0.7)
    
    # DE features
    plt.subplot(2, 2, 4)
    plt.plot(timestamps, results_df['de_mean'], 'm-d', label='DE Mean', linewidth=2)
    plt.plot(timestamps, results_df['de_std'], 'c-h', label='DE Std', linewidth=2)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Feature Value')
    plt.title('Neural Features')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📊 Session Summary:")
    print(f"   Average Valence: {np.mean(results_df['valence']):+.3f}")
    print(f"   Average Arousal: {np.mean(results_df['arousal']):+.3f}")
    print(f"   Average Φ: {np.mean(results_df['phi']):.6f}")
    print(f"   Φ Range: [{min(results_df['phi']):.6f}, {max(results_df['phi']):.6f}]")
    
    # Most common emotion
    emotion_counts = {}
    for v, a in zip(results_df['valence'], results_df['arousal']):
        emotion = simulator.get_emotion_label(v, a)
        emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1
    
    dominant_emotion = max(emotion_counts, key=emotion_counts.get)
    print(f"   Dominant Emotion: {dominant_emotion} ({emotion_counts[dominant_emotion]} updates)")

else:
    print("❌ No results to display")

## Step 9: Next Steps and Real Deployment

This notebook demonstrated the complete workflow. For real deployment:

In [ ]:
print("🎯 Next Steps for Real Deployment:")
print("=" * 40)
print()
print("1. 📊 Data Collection:")
print("   • Collect real EEG data with emotion labels")
print("   • Organize data in subject folders with labels.json files")
print("   • Use EEGLAB .set format for best compatibility")
print()
print("2. 🏋️ Model Training:")
print("   python scripts/train/train_labeled.py --data_dir data/training\\ set --epochs 100 --cv_method LOSO")
print("   python scripts/train/train_labeled.py --compute_phi --loss_fn CCC --model_type EfficientNet")
print()
print("3. 📤 Model Export:")
print("   python scripts/tools/export_onnx.py --weights model/model_weight/ckpt.pt")
print("   python scripts/tools/inference_benchmark.py --model model/model_onnx/va_regressor.onnx --compare")
print()
print("4. 🔴 Real-time Setup:")
print("   • Configure your EEG device (Muse2, OpenBCI, etc.)")
print("   • Start LSL streaming: python scripts/lsl_receiver.py --device Muse2")
print("   • Launch dashboard: npm run dev")
print()
print("5. 🧠 Advanced Features:")
print("   • Enable IIT Φ calculation: pip install -r requirements_phi.txt")
print("   • Cross-validation: --cv_method LOSO for subject-independent validation")
print("   • Custom devices: Add configuration to configs/device_mapping.json")
print()
print("6. 📱 Web Interface:")
print("   • Access dashboard at http://localhost:5000")
print("   • Real-time emotion visualization")
print("   • Historical data analysis")
print("   • Model performance monitoring")
print()
print("📚 Documentation:")
print("   • Hardware Guide: notebooks/HARDWARE_GUIDE.md")
print("   • API Documentation: Check server/index.ts")
print("   • Configuration: configs/ directory files")
print()
print("🎉 Congratulations! You've completed the Neural Axis BCI quick start.")
print("    The system is ready for real EEG emotion recognition!")